# Knee MRI v3 -- Laterality Normalization, Multi-Series Pooling, Region-Guided Fusion

v2 got a real, externally-validated **0.775 public score**. This version targets the gap to
the leaderboard's higher scorers with four specific, individually-justified changes -- not a
"more models" approach. See the conversation for why 12 per-pathology specialist models were
considered and rejected: real per-pathology SOTA (SGNET 0.97 AUC ACL, Keros 0.99 ACL / 0.96
medial meniscus, MRNet 0.965 ACL / 0.847 meniscus) all comes from hundreds-to-thousands of
professionally labeled cases *per pathology* -- we have 58 gold-labeled studies *total*.
Splitting the model would divide that budget further and throw away cross-label transfer
between correlated pathologies, which is the one thing working in our favor given how little
gold supervision exists.

## What's new, and why each one is grounded (not guessed)

1. **Laterality normalization** (Section A): right knees are horizontally mirrored so medial/
   lateral anatomy sits in a consistent position regardless of which knee was scanned. Taken
   from the real current best reference for this exact competition,
   [JunhaoLiXD/RSNA_Knee_Abnormality_Detection](https://github.com/JunhaoLiXD/RSNA_Knee_Abnormality_Detection)'s
   latest version. Verified the `Laterality` DICOM tag actually survives this dataset's
   metadata allowlist before building this -- present in 66/86 series checked directly
   against your local `Raw_dicom` (~77%), with real `'L'`/`'R'` values. Mirroring is a
   pragmatic column-flip, not a full geometric relabeling per imaging plane -- simpler, and
   consistent with how most published knee-MRI pipelines (MRNet-style) handle this; series
   with no laterality tag are left as acquired rather than guessed.
2. **Pool over ALL series in a plane, not just one priority series** (Section B): flagged as
   the biggest known gap in both v1 and v2's own "what's next" notes, never implemented until
   now. ~15% of study-plane pairs have 2+ series carrying real additional signal (a
   fluid-sensitive and a plain acquisition of the same plane) that v1/v2 discarded.
3. **Region-guided per-label attention over the 3 plane embeddings** (Section B), replacing
   flat concatenation -- the pathology-grouped design already sketched in
   `docs/model_architecture_plan.md`, and directionally the same idea as JunhaoLiXD's
   "target-specific attention-pooling head". Lets the model learn, per pathology-group, how
   much to trust each plane instead of always weighting all three equally.
4. **Per-label sample weighting from consensus disagreement** (Section D): `_disagreement`
   columns already exist in `train_folds_consensus.csv` (from the earlier label-aggregation
   work) and were never used for training. Rows/labels the consensus pipeline itself flagged
   as uncertain now get down-weighted in the loss instead of contributing full-strength noisy
   signal.

## Cost/compatibility note -- this is not a drop-in swap

Both the mirroring (changes Section A's pixel output) and multi-series pooling (changes the
cached feature *shape and content*, now `(3 x N_SLICES, 1280)` + a mask per plane instead of
`(N_SLICES, 1280)`) mean **the `cached-features-train-data` dataset you uploaded for v2 is not
compatible here** -- it doesn't have masks and only covers one series per plane. Section A and
B need a fresh full run over training data once, same as the very first v2 run. Recommend
running Section A + B locally again (you have the full `Raw_dicom` there) and uploading the
resulting `FEATURE_CACHE_DIR` as a new Kaggle dataset, exactly like before, rather than eating
this cost on Kaggle's clock. Section B will also take longer than v2's per this cache format
change alone (up to 3x the slices per plane) -- `PREPROCESS_BATCH_SIZE` is lowered accordingly.

In [1]:
import os
import time
import random
import threading
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import roc_auc_score
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import pydicom

TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture',
]
PLANES = ['Sagittal', 'Coronal', 'Axial']
DISAGREEMENT_COLS = [f'{c}__disagreement' for c in TARGET_COLS]

REGION_GROUPS = {
    'ligament': ['ACL', 'MCL'],
    'meniscus': ['Medial Meniscus', 'Lateral Meniscus'],
    'oa':       ['Medial OA', 'Lateral OA', 'PF OA'],
    'fluid':    ['Effusion', 'Synovitis', "Baker's"],
    'trauma':   ['Contusion', 'Fracture'],
}

In [2]:
# -----------------------------------------------------------------------
# Config -- IS_KAGGLE auto-detects the environment; only DATA_DIR needs a manual edit on
# Kaggle (point it at your dataset mount).
# -----------------------------------------------------------------------
IS_KAGGLE = os.path.isdir('/kaggle')

DATA_DIR = 'D:/Coding/kaggle/RSNA/data'   # <-- on Kaggle, change this to your dataset mount path
MAIN_DIR = f'{DATA_DIR}/main_data'

CONSENSUS_LABELS_CSV = f'{MAIN_DIR}/final/train_folds_consensus.csv'
SERIES_META_CSV = f'{MAIN_DIR}/train_series.csv'
GOLD_LABELS_CSV = f'{MAIN_DIR}/to_use/available_labels.csv'
TEST_CSV = f'{MAIN_DIR}/test.csv'
TEST_SERIES_CSV = f'{MAIN_DIR}/test_series.csv'
SAMPLE_SUB_CSV = f'{MAIN_DIR}/sample_submission.csv'

if IS_KAGGLE:
    KAGGLE_COMP_DIR = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
    TRAIN_DICOM_DIR = f'{KAGGLE_COMP_DIR}/train_series'
    TEST_DICOM_DIR = f'{KAGGLE_COMP_DIR}/test_series'
else:
    TRAIN_DICOM_DIR = f'{DATA_DIR}/Raw_dicom/train_series'
    TEST_DICOM_DIR = f'{DATA_DIR}/Raw_dicom/test_series'

OUTPUT_DIR = '/kaggle/working' if IS_KAGGLE else 'D:/Coding/kaggle/RSNA/output/v3'
PREPROCESSED_DIR = f'{OUTPUT_DIR}/preprocessed'          # Section A output: one (T,224,224) uint8 .npy per series
FEATURE_CACHE_DIR = f'{OUTPUT_DIR}/feature_cache'         # Section B output: one backbone-feature .npz per study
TEST_FEATURE_CACHE_DIR = f'{OUTPUT_DIR}/test_feature_cache'   # kept separate -- see v2's read-only-symlink lesson
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREPROCESSED_DIR, exist_ok=True)
os.makedirs(FEATURE_CACHE_DIR, exist_ok=True)
os.makedirs(TEST_FEATURE_CACHE_DIR, exist_ok=True)

# One-time link-up: symlink a pre-cached train-feature Kaggle Dataset into FEATURE_CACHE_DIR,
# if you've made one for THIS v3 format (see the compatibility note above -- the v2 cache
# will NOT work here, it lacks masks and multi-series pooling). Leave the path as-is if you
# haven't made one yet; Section B will just compute everything fresh.
TRAIN_FEATURE_CACHE_DATASET = '/kaggle/input/datasets/nishkarshsinghal/cached-features-train-data-v3/feature_cache'
if os.path.isdir(TRAIN_FEATURE_CACHE_DATASET):
    import shutil
    n_linked = 0
    for fname in os.listdir(TRAIN_FEATURE_CACHE_DATASET):
        dst = os.path.join(FEATURE_CACHE_DIR, fname)
        if os.path.exists(dst):
            continue
        src = os.path.join(TRAIN_FEATURE_CACHE_DATASET, fname)
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copy2(src, dst)
        n_linked += 1
    print(f'Linked {n_linked} pre-cached train features from {TRAIN_FEATURE_CACHE_DATASET}')
else:
    print(f'Pre-cached v3 train feature dataset not found at {TRAIN_FEATURE_CACHE_DATASET} '
          f'-- train features will be computed from scratch this run.')

# See v1's Section 2 note -- required so the internet-off Kaggle commit run doesn't crash
# trying to download pretrained weights.
EFFICIENTNET_WEIGHTS_PATH = None   # e.g. '/kaggle/input/datasets/.../efficientnet_b0_weights.pth'

# Data / model shape
N_SLICES = 12                 # slices sampled per series
MAX_SERIES_PER_PLANE = 3      # covers ~97% of study-plane pairs completely (measured on train_series.csv)
IMG_SIZE = 224
FREEZE_BACKBONE = True

# Preprocessing + precompute (one-time, CPU-heavy). Lowered vs. v2 -- each plane now processes
# up to MAX_SERIES_PER_PLANE x N_SLICES images instead of just N_SLICES.
PREPROCESS_BATCH_SIZE = 8
NUM_LOAD_WORKERS = max(1, (os.cpu_count() or 2))   # leave 1 core free for the OS/notebook

# Training (fast, on cached feature vectors)
TRAIN_BATCH_SIZE = 32
TRAIN_NUM_WORKERS = 0
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
GOLD_WEIGHT = 8.0
MIN_LABEL_CONFIDENCE = 0.2   # floor on the disagreement-derived per-label weight -- no label ever gets near-zero weight
USE_AMP = torch.cuda.is_available()

# [0] = fast sanity run. Use list(range(5)) for the real, full CV run.
FOLDS_TO_RUN = [0]

SEED = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | OUTPUT_DIR: {OUTPUT_DIR}")
print(f'Device: {device} | loader threads: {NUM_LOAD_WORKERS} (of {os.cpu_count()} logical cores)')

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

Pre-cached v3 train feature dataset not found at /kaggle/input/datasets/nishkarshsinghal/cached-features-train-data-v3/feature_cache -- train features will be computed from scratch this run.
Environment: Local | OUTPUT_DIR: D:/Coding/kaggle/RSNA/output/v3
Device: cuda | loader threads: 12 (of 12 logical cores)


## 1. Load series metadata (train + test)

In [3]:
train_series_df = pd.read_csv(SERIES_META_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
consensus_df = pd.read_csv(CONSENSUS_LABELS_CSV)
gold_df = pd.read_csv(GOLD_LABELS_CSV)
gold_ids = set(gold_df['StudyInstanceUID'])

print(f'Train studies: {consensus_df["StudyInstanceUID"].nunique()} | train series: {len(train_series_df)}')
print(f'Test studies: {test_series_df["StudyInstanceUID"].nunique()} | test series: {len(test_series_df)} '
      f'(the public stub -- real run substitutes ~1300 studies here)')
print(f'Gold rows: {len(gold_ids)}')
print(f'Disagreement columns present in consensus file: {sum(c in consensus_df.columns for c in DISAGREEMENT_COLS)}/12')


def build_all_series_by_plane(series_meta, max_per_plane=MAX_SERIES_PER_PLANE):
    """(study, plane) -> up to max_per_plane SeriesInstanceUIDs, fluid-sensitive series first."""
    result = {}
    series_meta = series_meta.sort_values(['Fluid_Sensitive', 'SeriesInstanceUID'], ascending=[False, True])
    for (uid, plane), grp in series_meta.groupby(['StudyInstanceUID', 'Anatomical_Plane']):
        result[(uid, plane)] = grp['SeriesInstanceUID'].tolist()[:max_per_plane]
    return result


train_series_by_plane = build_all_series_by_plane(train_series_df)
test_series_by_plane = build_all_series_by_plane(test_series_df)
n_train_multi = sum(len(v) > 1 for v in train_series_by_plane.values())
print(f'Train study-plane groups: {len(train_series_by_plane)} ({n_train_multi} have 2+ series -- '
      f'these are exactly what v1/v2 discarded)')

Train studies: 4407 | train series: 24371
Test studies: 3 | test series: 15 (the public stub -- real run substitutes ~1300 studies here)
Gold rows: 58
Disagreement columns present in consensus file: 12/12
Train study-plane groups: 13221 (8665 have 2+ series -- these are exactly what v1/v2 discarded)


## Section A: DICOM preprocessing (+ laterality normalization)

Same core decode as v2 (spatially-correct slice ordering via `ImagePositionPatient`
projection, rescale/photometric correction, per-series percentile normalization) with one
addition: series from a right knee (`Laterality` == `'R'`) are horizontally mirrored so
medial/lateral anatomy sits in a consistent position regardless of which knee was imaged.
Series with no laterality tag are left as acquired -- not guessed.

In [4]:
def _order_series_files(series_dir):
    """Return [(filename, pydicom.Dataset), ...] in spatially-correct order."""
    fnames = [f for f in os.listdir(series_dir) if f.lower().endswith('.dcm')]
    entries = []
    for fname in fnames:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, fname))
        except Exception:
            continue
        entries.append([None, fname, ds])
    if not entries:
        return []

    have_geometry = True
    for e in entries:
        try:
            iop = np.array(e[2].ImageOrientationPatient, dtype=np.float64)
            ipp = np.array(e[2].ImagePositionPatient, dtype=np.float64)
            normal = np.cross(iop[0:3], iop[3:6])
            e[0] = float(np.dot(normal, ipp))
        except Exception:
            have_geometry = False
            break
    if have_geometry and len({round(e[0], 3) for e in entries}) > 1:
        entries.sort(key=lambda e: e[0])
        return [(e[1], e[2]) for e in entries]

    try:
        nums = [int(getattr(e[2], 'InstanceNumber', -1)) for e in entries]
        if len(set(nums)) == len(entries) and all(n >= 0 for n in nums):
            order = sorted(range(len(entries)), key=lambda i: nums[i])
            return [(entries[i][1], entries[i][2]) for i in order]
    except Exception:
        pass

    entries.sort(key=lambda e: e[1])
    return [(e[1], e[2]) for e in entries]


def _decode_slice(ds):
    arr = ds.pixel_array.astype(np.float32)
    if arr.ndim == 3:
        arr = arr[arr.shape[0] // 2]
    slope = float(getattr(ds, 'RescaleSlope', 1))
    intercept = float(getattr(ds, 'RescaleIntercept', 0))
    arr = arr * slope + intercept
    if str(getattr(ds, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
        arr = arr.max() - arr
    return arr


def _tag(ds, name, cast=None, default=None):
    if not hasattr(ds, name):
        return default
    try:
        val = getattr(ds, name)
        return cast(val) if cast else val
    except Exception:
        return default


def preprocess_series(series_dir, img_size, anatomical_plane=None):
    """One series directory -> ((T, img_size, img_size) uint8 array, metadata dict) or (None, {})."""
    ordered = _order_series_files(series_dir)
    if not ordered:
        return None, {}

    slices = []
    for _, ds in ordered:
        try:
            slices.append(_decode_slice(ds))
        except Exception:
            continue
    if not slices:
        return None, {}

    stack = np.stack(slices, axis=0)
    lo, hi = np.percentile(stack, [1.0, 99.0])
    hi = max(hi, lo + 1.0)
    stack = np.clip((stack - lo) / (hi - lo), 0, 1) * 255.0
    stack = stack.astype(np.uint8)

    first_ds = ordered[0][1]
    laterality = str(_tag(first_ds, 'Laterality', str, '') or '').strip().upper()
    mirrored = False
    # Only Coronal/Axial have an image-column axis that actually corresponds to true patient
    # left-right -- a Sagittal slice's horizontal axis is anterior-posterior, unrelated to which
    # knee it is. Mirroring Sagittal by Laterality was flipping anterior<->posterior for ~half
    # the sagittal images for no anatomically valid reason (this was v3's real regression: ACL,
    # diagnosed almost entirely from sagittal sequences, dropped hardest).
    if laterality == 'R' and anatomical_plane in ('Coronal', 'Axial'):
        stack = stack[:, :, ::-1]   # mirror left-right -- consistent medial/lateral position
        mirrored = True

    resized = np.stack([
        np.array(Image.fromarray(np.ascontiguousarray(s)).resize((img_size, img_size), Image.BILINEAR), dtype=np.uint8)
        for s in stack
    ])

    spacing = _tag(first_ds, 'PixelSpacing')
    meta = {
        'n_dcm_files': len(ordered),
        'n_slices_decoded': len(slices),
        'native_rows': _tag(first_ds, 'Rows', int),
        'native_cols': _tag(first_ds, 'Columns', int),
        'pixel_spacing_row': float(spacing[0]) if spacing is not None else None,
        'pixel_spacing_col': float(spacing[1]) if spacing is not None else None,
        'slice_thickness': _tag(first_ds, 'SliceThickness', float),
        'manufacturer': _tag(first_ds, 'Manufacturer', str, ''),
        'manufacturer_model': _tag(first_ds, 'ManufacturerModelName', str, ''),
        'magnetic_field_strength': _tag(first_ds, 'MagneticFieldStrength', float),
        'photometric_interpretation': _tag(first_ds, 'PhotometricInterpretation', str, ''),
        'laterality': laterality,
        'mirrored': mirrored,
    }
    return resized, meta

In [5]:
def preprocess_all_series(series_df, dicom_root, out_dir, img_size, num_workers, skip_existing=True):
    rows = list(series_df.itertuples())
    todo = [r for r in rows
            if not (skip_existing and os.path.exists(os.path.join(out_dir, f'{r.SeriesInstanceUID}.npy')))]
    print(f'{len(todo)} / {len(rows)} series need preprocessing ({num_workers} threads) -> {dicom_root}')
    if not todo:
        return []

    def _work(row):
        series_dir = os.path.join(dicom_root, row.StudyInstanceUID, row.SeriesInstanceUID)
        if not os.path.isdir(series_dir):
            return row.SeriesInstanceUID, None, {}
        arr, meta = preprocess_series(series_dir, img_size, anatomical_plane=row.Anatomical_Plane)
        return row.SeriesInstanceUID, arr, meta

    meta_rows = []
    t0 = time.time()
    done = 0
    failed = 0
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        for series_uid, arr, meta in pool.map(_work, todo):
            if arr is not None:
                np.save(os.path.join(out_dir, f'{series_uid}.npy'), arr)
            else:
                failed += 1
            meta['SeriesInstanceUID'] = series_uid
            meta['ok'] = arr is not None
            meta_rows.append(meta)
            done += 1
            if done % 200 == 0 or done == len(todo):
                elapsed = time.time() - t0
                rate = done / elapsed if elapsed > 0 else 0.0
                print(f'  {done}/{len(todo)} | {elapsed:.0f}s | {rate:.2f} series/s | {failed} failed so far')

    print(f'Done: {done} series in {time.time() - t0:.0f}s, {failed} failed -> {out_dir}')
    if failed > 0:
        print(f'WARNING: {failed} series produced no usable slices -- studies relying on them '
              f'will have that series masked out in Section B (not silently zeroed into the average).')
    return meta_rows

In [6]:
# Run once (Section A): preprocesses every train + test series into PREPROCESSED_DIR.
# Skips train series belonging to studies whose features are already cached (via the
# symlinks above), same optimization as v2 -- only touches what Section B doesn't already have.
train_studies_with_features = {f[:-4] for f in os.listdir(FEATURE_CACHE_DIR) if f.endswith('.npz')}
train_series_todo = train_series_df[~train_series_df['StudyInstanceUID'].isin(train_studies_with_features)]
print(f'{len(train_series_todo)} / {len(train_series_df)} train series need preprocessing '
      f'({len(train_studies_with_features)} studies already have cached features, skipped)')

train_meta = preprocess_all_series(train_series_todo, TRAIN_DICOM_DIR, PREPROCESSED_DIR,
                                    IMG_SIZE, NUM_LOAD_WORKERS)
test_meta = preprocess_all_series(test_series_df, TEST_DICOM_DIR, PREPROCESSED_DIR,
                                   IMG_SIZE, NUM_LOAD_WORKERS, skip_existing=False)

meta_df = pd.DataFrame(train_meta + test_meta)
if len(meta_df):
    meta_path = os.path.join(PREPROCESSED_DIR, 'preprocessing_metadata.csv')
    meta_df.to_csv(meta_path, index=False)
    print(f"\nWrote metadata for {len(meta_df)} series -> {meta_path}")
    print(meta_df['ok'].value_counts())
    if 'laterality' in meta_df.columns:
        print('\nLaterality tag coverage:')
        print(meta_df['laterality'].value_counts(dropna=False))

24371 / 24371 train series need preprocessing (0 studies already have cached features, skipped)
9808 / 24371 series need preprocessing (12 threads) -> D:/Coding/kaggle/RSNA/data/Raw_dicom/train_series
  200/9808 | 45s | 4.42 series/s | 0 failed so far
  400/9808 | 88s | 4.55 series/s | 0 failed so far
  600/9808 | 119s | 5.03 series/s | 0 failed so far
  800/9808 | 168s | 4.75 series/s | 0 failed so far
  1000/9808 | 207s | 4.83 series/s | 0 failed so far
  1200/9808 | 220s | 5.46 series/s | 0 failed so far
  1400/9808 | 255s | 5.49 series/s | 0 failed so far
  1600/9808 | 294s | 5.45 series/s | 0 failed so far
  1800/9808 | 346s | 5.20 series/s | 0 failed so far
  2000/9808 | 367s | 5.45 series/s | 0 failed so far
  2200/9808 | 405s | 5.43 series/s | 0 failed so far
  2400/9808 | 431s | 5.57 series/s | 0 failed so far
  2600/9808 | 474s | 5.49 series/s | 0 failed so far
  2800/9808 | 495s | 5.66 series/s | 0 failed so far
  3000/9808 | 535s | 5.61 series/s | 0 failed so far
  3200/980

## Section B: precompute frozen-backbone features (all series per plane, masked)

Each plane now aggregates up to `MAX_SERIES_PER_PLANE` series' slices into one padded,
masked bag (`MAX_SERIES_PER_PLANE * N_SLICES` slots) instead of a single series' `N_SLICES` --
covering the ~15% of study-plane pairs where v1/v2 silently dropped a second acquisition.
`AttentionPool` is now mask-aware so padding never pollutes the pooled average.

In [7]:
class SliceBackbone(nn.Module):
    def __init__(self, freeze=True):
        super().__init__()
        if EFFICIENTNET_WEIGHTS_PATH and os.path.exists(EFFICIENTNET_WEIGHTS_PATH):
            m = efficientnet_b0(weights=None)
            m.load_state_dict(torch.load(EFFICIENTNET_WEIGHTS_PATH, map_location='cpu'))
        else:
            m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)   # needs internet
        self.features = m.features
        self.pool = m.avgpool
        self.out_dim = 1280
        if freeze:
            for p in self.features.parameters():
                p.requires_grad = False

    def forward(self, x):        # x: (N, 3, H, W)
        f = self.features(x)
        return self.pool(f).flatten(1)   # (N, out_dim)


class AttentionPool(nn.Module):
    """Mask-aware attention pooling -- padded (all-zero) slots never contribute to the average."""

    def __init__(self, dim, hidden=128):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):        # x: (B, T, dim), mask: (B, T) 1=valid, 0=pad
        scores = self.attn(x)               # (B, T, 1)
        if mask is not None:
            invalid = (mask == 0).unsqueeze(-1)
            all_invalid = invalid.all(dim=1, keepdim=True)   # guard: never let a whole sample go to NaN
            scores = scores.masked_fill(invalid & ~all_invalid, float('-inf'))
        w = torch.softmax(scores, dim=1)
        return (x * w).sum(dim=1)


class PlaneAttentionHead(nn.Module):
    """One anatomical-region group: learns how much to trust each of the 3 planes, then classifies."""

    def __init__(self, dim, num_targets, hidden=128):
        super().__init__()
        self.plane_attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
        self.classifier = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(hidden, num_targets)
        )

    def forward(self, plane_embeds):        # (B, 3, dim)
        w = torch.softmax(self.plane_attn(plane_embeds), dim=1)   # (B, 3, 1)
        fused = (plane_embeds * w).sum(dim=1)                       # (B, dim)
        return self.classifier(fused)


class RegionGuidedHead(nn.Module):
    def __init__(self, dim, region_groups, target_cols):
        super().__init__()
        self.target_cols = target_cols
        self.heads = nn.ModuleDict()
        self.target_indices = {}
        for name, targets in region_groups.items():
            self.heads[name] = PlaneAttentionHead(dim, len(targets))
            self.target_indices[name] = [target_cols.index(t) for t in targets]

    def forward(self, plane_embeds):         # (B, 3, dim)
        B = plane_embeds.size(0)
        logits = torch.zeros(B, len(self.target_cols), device=plane_embeds.device, dtype=plane_embeds.dtype)
        for name, head in self.heads.items():
            out = head(plane_embeds)
            for i, idx in enumerate(self.target_indices[name]):
                logits[:, idx] = out[:, i]
        return logits

In [8]:
_fallback_lock = threading.Lock()
_fallback_count = [0]


def _load_plane_all_series(uid, plane, series_by_plane):
    """(MAX_SERIES_PER_PLANE * N_SLICES, H, W) float32 in [0,1] + matching (same length,) mask."""
    series_list = series_by_plane.get((uid, plane), [])
    arrs, masks = [], []
    for series_uid in series_list[:MAX_SERIES_PER_PLANE]:
        path = os.path.join(PREPROCESSED_DIR, f'{series_uid}.npy')
        if not os.path.exists(path):
            with _fallback_lock:
                _fallback_count[0] += 1
            continue
        arr = np.load(path).astype(np.float32) / 255.0
        T = arr.shape[0]
        if T > N_SLICES:
            idx = np.linspace(0, T - 1, N_SLICES).astype(int)
            arr = arr[idx]
            m = np.ones(N_SLICES, dtype=np.float32)
        elif T < N_SLICES:
            pad = np.zeros((N_SLICES - T, IMG_SIZE, IMG_SIZE), dtype=np.float32)
            m = np.concatenate([np.ones(T, dtype=np.float32), np.zeros(N_SLICES - T, dtype=np.float32)])
            arr = np.concatenate([arr, pad], axis=0)
        else:
            m = np.ones(N_SLICES, dtype=np.float32)
        arrs.append(arr)
        masks.append(m)

    while len(arrs) < MAX_SERIES_PER_PLANE:
        arrs.append(np.zeros((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.float32))
        masks.append(np.zeros(N_SLICES, dtype=np.float32))

    return np.concatenate(arrs, axis=0), np.concatenate(masks, axis=0)


def _load_study_planes(uid, series_by_plane):
    out = {}
    for plane, key in zip(PLANES, ['sagittal', 'coronal', 'axial']):
        arr, mask = _load_plane_all_series(uid, plane, series_by_plane)
        out[key] = arr
        out[f'{key}_mask'] = mask
    return uid, out


@torch.no_grad()
def precompute_features(study_ids, series_by_plane, backbone, cache_dir,
                         batch_size=PREPROCESS_BATCH_SIZE, num_load_workers=NUM_LOAD_WORKERS,
                         skip_existing=True):
    todo = [u for u in study_ids if not (skip_existing and os.path.exists(os.path.join(cache_dir, f'{u}.npz')))]
    print(f'{len(todo)} / {len(study_ids)} studies need feature extraction '
          f'({num_load_workers} loader threads, batch size {batch_size})')
    if not todo:
        return

    backbone.eval()
    t0 = time.time()
    done = 0
    _fallback_count[0] = 0

    with ThreadPoolExecutor(max_workers=num_load_workers) as pool:
        for start in range(0, len(todo), batch_size):
            batch_ids = todo[start: start + batch_size]
            results = list(pool.map(lambda u: _load_study_planes(u, series_by_plane), batch_ids))

            feats = {}
            for key in ['sagittal', 'coronal', 'axial']:
                raw = np.stack([r[1][key] for r in results])                  # (B, T, H, W)
                x = torch.from_numpy(raw).unsqueeze(2).repeat(1, 1, 3, 1, 1)   # (B, T, 3, H, W)
                x = (x - IMAGENET_MEAN) / IMAGENET_STD
                x = x.to(device)
                B, T, C, H, W = x.shape
                feats[key] = backbone(x.view(B * T, C, H, W)).view(B, T, -1).cpu().numpy()

            for b, (uid, planes) in enumerate(results):
                np.savez(os.path.join(cache_dir, f'{uid}.npz'),
                         sagittal=feats['sagittal'][b], sagittal_mask=planes['sagittal_mask'],
                         coronal=feats['coronal'][b], coronal_mask=planes['coronal_mask'],
                         axial=feats['axial'][b], axial_mask=planes['axial_mask'])

            done += len(batch_ids)
            if (start // batch_size) % 20 == 0 or done == len(todo):
                elapsed = time.time() - t0
                rate = done / elapsed if elapsed > 0 else 0.0
                print(f'  {done}/{len(todo)} studies | {elapsed:.0f}s elapsed | {rate:.2f} studies/s')

    print(f'Precompute finished: {done} studies in {time.time() - t0:.0f}s -> {cache_dir}')
    if _fallback_count[0] > 0:
        print(f'Note: {_fallback_count[0]} series-slots were missing from PREPROCESSED_DIR and are masked '
              f'out (not silently zeroed into the average) -- check Section A\'s failure count if this is high.')

In [9]:
class KneeFeatureDataset(Dataset):
    """Loads cached per-plane backbone features + masks for one study, plus per-label
    training weight derived from consensus disagreement (confidence = 1 - disagreement,
    floored at MIN_LABEL_CONFIDENCE so no label ever gets near-zero weight)."""

    def __init__(self, study_ids, cache_dir, labels_df=None, gold_ids=None, gold_weight=1.0):
        self.study_ids = list(study_ids)
        self.cache_dir = cache_dir
        self.gold_ids = gold_ids or set()
        self.gold_weight = gold_weight
        self.label_lookup = None
        self.label_weight_lookup = None
        if labels_df is not None:
            idx = labels_df.set_index('StudyInstanceUID')[TARGET_COLS]
            self.label_lookup = {uid: idx.loc[uid].astype(float).values for uid in self.study_ids}
            if all(c in labels_df.columns for c in DISAGREEMENT_COLS):
                dis = labels_df.set_index('StudyInstanceUID')[DISAGREEMENT_COLS].clip(lower=0.0, upper=1.0)
                conf = (1.0 - dis).clip(lower=MIN_LABEL_CONFIDENCE, upper=1.0)
                self.label_weight_lookup = {uid: conf.loc[uid].astype(float).values for uid in self.study_ids}

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        uid = self.study_ids[idx]
        d = np.load(os.path.join(self.cache_dir, f'{uid}.npz'))
        item = {'study_uid': uid}
        for key in ['sagittal', 'coronal', 'axial']:
            item[key] = torch.from_numpy(d[key]).float()
            item[f'{key}_mask'] = torch.from_numpy(d[f'{key}_mask']).float()

        if self.label_lookup is not None:
            item['labels'] = torch.tensor(self.label_lookup[uid], dtype=torch.float32)
            item['weight'] = torch.tensor(self.gold_weight if uid in self.gold_ids else 1.0, dtype=torch.float32)
            if self.label_weight_lookup is not None:
                item['label_weight'] = torch.tensor(self.label_weight_lookup[uid], dtype=torch.float32)
            else:
                item['label_weight'] = torch.ones(len(TARGET_COLS), dtype=torch.float32)
        return item


class KneeFeatureModel(nn.Module):
    def __init__(self, feat_dim=1280, target_cols=TARGET_COLS, region_groups=REGION_GROUPS):
        super().__init__()
        self.slice_pool = AttentionPool(feat_dim)
        self.region_head = RegionGuidedHead(feat_dim, region_groups, target_cols)

    def forward(self, sagittal, coronal, axial, sag_mask=None, cor_mask=None, ax_mask=None):
        s = self.slice_pool(sagittal, sag_mask)
        c = self.slice_pool(coronal, cor_mask)
        a = self.slice_pool(axial, ax_mask)
        plane_embeds = torch.stack([s, c, a], dim=1)   # (B, 3, dim)
        return self.region_head(plane_embeds)

In [10]:
# Run once (Section B): precomputes + caches backbone features for every labeled study.
backbone = SliceBackbone(freeze=FREEZE_BACKBONE).to(device)
precompute_features(consensus_df['StudyInstanceUID'].tolist(), train_series_by_plane, backbone, FEATURE_CACHE_DIR)

4407 / 4407 studies need feature extraction (12 loader threads, batch size 8)
  8/4407 studies | 2s elapsed | 3.59 studies/s
  168/4407 studies | 35s elapsed | 4.85 studies/s
  328/4407 studies | 67s elapsed | 4.91 studies/s
  488/4407 studies | 99s elapsed | 4.91 studies/s
  648/4407 studies | 132s elapsed | 4.91 studies/s
  808/4407 studies | 165s elapsed | 4.90 studies/s
  968/4407 studies | 198s elapsed | 4.89 studies/s
  1128/4407 studies | 230s elapsed | 4.90 studies/s
  1288/4407 studies | 263s elapsed | 4.90 studies/s
  1448/4407 studies | 296s elapsed | 4.90 studies/s
  1608/4407 studies | 328s elapsed | 4.90 studies/s
  1768/4407 studies | 361s elapsed | 4.90 studies/s
  1928/4407 studies | 394s elapsed | 4.89 studies/s
  2088/4407 studies | 427s elapsed | 4.89 studies/s
  2248/4407 studies | 459s elapsed | 4.89 studies/s
  2408/4407 studies | 492s elapsed | 4.89 studies/s
  2568/4407 studies | 525s elapsed | 4.89 studies/s
  2728/4407 studies | 558s elapsed | 4.89 studies/s


## Section C: loss & metrics

In [11]:
def weighted_bce(logits, targets, row_weight, label_weight, pos_weight):
    per_elem = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction='none')  # (B,12)
    weighted_elem = per_elem * label_weight                                                                   # (B,12)
    per_sample = weighted_elem.mean(dim=1)
    return (per_sample * row_weight).sum() / row_weight.sum()


def macro_auc(y_true_bin, y_prob, cols):
    """y_true_bin: (N, len(cols)) hard 0/1 labels. Skips columns with only one class present."""
    per_class = {}
    for i, c in enumerate(cols):
        yt = y_true_bin[:, i]
        if len(np.unique(yt)) < 2:
            continue
        per_class[c] = roc_auc_score(yt, y_prob[:, i])
    macro = float(np.mean(list(per_class.values()))) if per_class else float('nan')
    return macro, per_class

## Section D: train / eval loop functions

In [12]:
def _to_device(batch):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def train_one_epoch(model, loader, optimizer, pos_weight, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        b = _to_device(batch)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(b['sagittal'], b['coronal'], b['axial'],
                           b['sagittal_mask'], b['coronal_mask'], b['axial_mask'])
            loss = weighted_bce(logits, b['labels'], b['weight'], b['label_weight'], pos_weight)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * b['sagittal'].size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs, all_uids = [], []
    for batch in loader:
        b = _to_device(batch)
        logits = model(b['sagittal'], b['coronal'], b['axial'],
                       b['sagittal_mask'], b['coronal_mask'], b['axial_mask'])
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_uids.extend(batch['study_uid'])
    return np.concatenate(all_probs, axis=0), all_uids

## Section E: run training (per fold)

In [13]:
consensus_indexed = consensus_df.set_index('StudyInstanceUID')
gold_indexed = gold_df.set_index('StudyInstanceUID')

oof_probs = {}
fold_ckpts = {}

for fold in FOLDS_TO_RUN:
    print(f"\n{'=' * 72}\nFold {fold}\n{'=' * 72}")
    train_ids = consensus_df.loc[consensus_df.fold != fold, 'StudyInstanceUID'].tolist()
    val_ids = consensus_df.loc[consensus_df.fold == fold, 'StudyInstanceUID'].tolist()
    print(f'train studies: {len(train_ids)} | val studies: {len(val_ids)}')

    train_ds = KneeFeatureDataset(train_ids, FEATURE_CACHE_DIR,
                                   labels_df=consensus_df, gold_ids=gold_ids, gold_weight=GOLD_WEIGHT)
    val_ds = KneeFeatureDataset(val_ids, FEATURE_CACHE_DIR,
                                 labels_df=consensus_df, gold_ids=gold_ids, gold_weight=GOLD_WEIGHT)

    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                               num_workers=TRAIN_NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
                             num_workers=TRAIN_NUM_WORKERS, pin_memory=True)

    prevalence = consensus_df.loc[consensus_df.fold != fold, TARGET_COLS].ge(0.5).mean().values
    pos_weight = torch.tensor(np.clip((1 - prevalence) / np.clip(prevalence, 1e-3, None), 1.0, 10.0),
                               dtype=torch.float32).to(device)

    model = KneeFeatureModel(feat_dim=1280).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_gold_auc = -1.0
    ckpt_path = os.path.join(OUTPUT_DIR, f'v3_fold{fold}.pt')

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, pos_weight, scaler)
        val_probs, val_uids = predict(model, val_loader)

        val_soft_true = consensus_indexed.loc[val_uids, TARGET_COLS].values
        noisy_auc, _ = macro_auc((val_soft_true >= 0.5).astype(int), val_probs, TARGET_COLS)

        gold_mask = np.array([u in gold_ids for u in val_uids])
        if gold_mask.any():
            gold_uids_here = [u for u, m in zip(val_uids, gold_mask) if m]
            gold_true = gold_indexed.loc[gold_uids_here, TARGET_COLS].values.astype(int)
            gold_auc, _ = macro_auc(gold_true, val_probs[gold_mask], TARGET_COLS)
        else:
            gold_auc = float('nan')

        marker = ''
        if not np.isnan(gold_auc) and gold_auc > best_gold_auc:
            best_gold_auc = gold_auc
            marker = ' *'
            torch.save(model.state_dict(), ckpt_path)

        print(f'epoch {epoch:>2}/{EPOCHS} | loss {train_loss:.4f} | '
              f'noisy AUC {noisy_auc:.4f} | gold AUC {gold_auc:.4f} | '
              f'{time.time() - t0:>5.1f}s{marker}')

    if not os.path.exists(ckpt_path):
        torch.save(model.state_dict(), ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    val_probs, val_uids = predict(model, val_loader)
    for u, p in zip(val_uids, val_probs):
        oof_probs[u] = p
    fold_ckpts[fold] = ckpt_path
    print(f'Fold {fold} best gold AUC: {best_gold_auc:.4f} -> {ckpt_path}')


Fold 0
train studies: 3525 | val studies: 882


C:\Users\nishk\AppData\Local\Temp\ipykernel_14028\2963079106.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


epoch  1/20 | loss 0.7891 | noisy AUC 0.7230 | gold AUC 0.6965 |  46.0s *
epoch  2/20 | loss 0.7513 | noisy AUC 0.7476 | gold AUC 0.7031 |   8.2s *
epoch  3/20 | loss 0.7307 | noisy AUC 0.7618 | gold AUC 0.6735 |   7.8s
epoch  4/20 | loss 0.7143 | noisy AUC 0.7705 | gold AUC 0.7294 |   7.9s *
epoch  5/20 | loss 0.7024 | noisy AUC 0.7689 | gold AUC 0.7277 |   7.9s
epoch  6/20 | loss 0.6881 | noisy AUC 0.7694 | gold AUC 0.7183 |   7.9s
epoch  7/20 | loss 0.6776 | noisy AUC 0.7737 | gold AUC 0.7288 |   7.7s
epoch  8/20 | loss 0.6661 | noisy AUC 0.7667 | gold AUC 0.7184 |   7.7s
epoch  9/20 | loss 0.6564 | noisy AUC 0.7685 | gold AUC 0.6849 |   7.7s
epoch 10/20 | loss 0.6438 | noisy AUC 0.7697 | gold AUC 0.7001 |   7.8s
epoch 11/20 | loss 0.6320 | noisy AUC 0.7653 | gold AUC 0.7052 |   7.5s
epoch 12/20 | loss 0.6210 | noisy AUC 0.7609 | gold AUC 0.7042 |   7.6s
epoch 13/20 | loss 0.6116 | noisy AUC 0.7617 | gold AUC 0.6781 |   7.7s
epoch 14/20 | loss 0.6000 | noisy AUC 0.7546 | gold AUC 0.

## Section F: out-of-fold summary

In [14]:
oof_uids = list(oof_probs.keys())
oof_matrix = np.stack([oof_probs[u] for u in oof_uids])

oof_soft_true = consensus_indexed.loc[oof_uids, TARGET_COLS].values
noisy_macro, noisy_per_class = macro_auc((oof_soft_true >= 0.5).astype(int), oof_matrix, TARGET_COLS)

gold_mask = np.array([u in gold_ids for u in oof_uids])
gold_oof_uids = [u for u, m in zip(oof_uids, gold_mask) if m]
gold_true = gold_indexed.loc[gold_oof_uids, TARGET_COLS].values.astype(int)
gold_macro, gold_per_class = macro_auc(gold_true, oof_matrix[gold_mask], TARGET_COLS)

print(f'OOF studies: {len(oof_uids)} (of which genuinely-labeled: {len(gold_oof_uids)})')
print(f'Noisy macro AUC  (vs. consensus soft pseudo-labels): {noisy_macro:.4f}')
print(f'HONEST macro AUC (vs. the 58 real gold labels):      {gold_macro:.4f}   <-- compare against v2\'s 0.7259 / real LB 0.775')

print('\nPer-class honest AUC:')
for c, v in sorted(gold_per_class.items(), key=lambda x: -x[1]):
    print(f'  {c:<20} {v:.4f}')

OOF studies: 882 (of which genuinely-labeled: 13)
Noisy macro AUC  (vs. consensus soft pseudo-labels): 0.7705
HONEST macro AUC (vs. the 58 real gold labels):      0.7294   <-- compare against v2's 0.7259 / real LB 0.775

Per-class honest AUC:
  Lateral OA           1.0000
  Contusion            0.9524
  Medial OA            0.8611
  Fracture             0.8333
  Effusion             0.8056
  ACL                  0.7222
  Baker's              0.6944
  MCL                  0.6364
  Synovitis            0.5952
  Lateral Meniscus     0.5714
  Medial Meniscus      0.5476
  PF OA                0.5333


## Section G: inference on test set + submission file

Test features are written to `TEST_FEATURE_CACHE_DIR`, kept separate from the (possibly
symlinked) `FEATURE_CACHE_DIR` -- see v2's read-only-filesystem lesson.

In [ ]:
test_df = pd.read_csv(TEST_CSV)

precompute_features(test_df['StudyInstanceUID'].tolist(), test_series_by_plane, backbone, TEST_FEATURE_CACHE_DIR,
                    skip_existing=False)

test_ds = KneeFeatureDataset(test_df['StudyInstanceUID'].tolist(), TEST_FEATURE_CACHE_DIR)
test_loader = DataLoader(test_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=TRAIN_NUM_WORKERS)

fold_preds = []
test_uids = None
for fold, ckpt_path in fold_ckpts.items():
    model = KneeFeatureModel(feat_dim=1280).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    probs, uids = predict(model, test_loader)
    fold_preds.append(probs)
    test_uids = uids

final_probs = np.mean(fold_preds, axis=0)
sub = pd.DataFrame(final_probs, columns=TARGET_COLS)
sub.insert(0, 'StudyInstanceUID', test_uids)
sub = sub[pd.read_csv(SAMPLE_SUB_CSV).columns]   # match exact column order

sub_path = os.path.join(OUTPUT_DIR, 'submission.csv')   # Kaggle requires this exact filename
sub.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
sub.head()

3 / 3 studies need feature extraction (12 loader threads, batch size 8)
  3/3 studies | 1s elapsed | 3.98 studies/s
Precompute finished: 3 studies in 1s -> D:/Coding/kaggle/RSNA/output/v3/test_feature_cache
Saved: D:/Coding/kaggle/RSNA/output/v3\submission.csv


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.213738,0.334839,0.337291,0.302417,0.319138,0.248697,0.285222,0.384591,0.377705,0.417597,0.437812,0.430402
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.680984,0.566216,0.379052,0.333963,0.587375,0.570999,0.503684,0.610672,0.875907,0.530136,0.552433,0.502378
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.150701,0.323524,0.331963,0.266844,0.329818,0.280859,0.341385,0.291424,0.332851,0.358503,0.327054,0.409632


: 

## What v4 should add

- Unfreeze the last block or two of the backbone once this converges (Phase 2 fine-tuning) --
  the single biggest lever not yet pulled across v1/v2/v3, all of which kept the backbone
  fully frozen.
- Swap EfficientNet-B0 for DINOv2-Small (matches JunhaoLiXD's latest version) -- needs the
  same internet-independent-weights treatment as EfficientNet-B0 got in v1/v2.
- If `preprocessing_metadata.csv`'s laterality coverage turns out low on the real Kaggle
  DICOM (only checked locally against train data here), consider a geometry-based fallback
  (patient position + orientation) instead of relying on the tag alone.
- Ensemble multiple full joint models (different seeds/backbones) rather than fragmenting
  into per-pathology specialists -- the standard, proven way to gain a few more AUC points
  without repeating the 58-label data-scarcity mistake discussed above.